In [ ]:
from datasets import load_dataset
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os
from google.colab import drive

In [ ]:
# --- 2. Khởi tạo Spark với cấu hình TỐI ƯU (High Performance) ---
spark = SparkSession.builder \
    .appName(f"Amazon_Bronze_ETL") \
    .config("spark.master", "local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.sql.shuffle.partitions", "2") \
    .config("spark.sql.parquet.int96RebaseModeInRead", "LEGACY") \
    .config("spark.sql.parquet.datetimeRebaseModeInRead", "LEGACY") \
    .getOrCreate()

In [ ]:
# --- 3. Load Dataset từ Hugging Face (Streaming Mode) ---
dataset = load_dataset(
    "json",
    data_files=f"https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/{CATEGORY}.jsonl",
    split="train",
    streaming=True
)

In [ ]:
# --- 1. Mount Drive & Cấu hình đường dẫn ---
drive.mount('/content/drive')

CATEGORY = "All_Beauty"
DATA_TYPE = "review"
BASE_PATH = "/content/drive/MyDrive/bigData/bronze"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def flow(CATEGORY, DATA_TYPE, BASE_PATH,batch=200000):
    FINAL_SAVE_PATH = os.path.join(BASE_PATH, DATA_TYPE, CATEGORY)
    os.makedirs(FINAL_SAVE_PATH, exist_ok=True)
    # --- 4. Vòng lặp Batching 50.000 dòng ---
    BATCH_SIZE = batch
    buffer = []

    print(f"🚀 Bắt đầu đẩy dữ liệu vào tầng Bronze: {CATEGORY}")

    for i, row in enumerate(dataset):
        # Chỉ giữ lại các cột cần thiết (Đã loại bỏ title)
        keep_keys = ['rating', 'text', 'parent_asin', 'timestamp', 'user_id', 'helpful_vote', 'verified_purchase']
        buffer.append({k: row.get(k) for k in keep_keys})

        if len(buffer) == BATCH_SIZE:
            # A. Chuyển đổi sang Spark DF (Sử dụng Arrow cực nhanh)
            df_batch = spark.createDataFrame(buffer)

            # B. Làm sạch kỹ thuật & Ép kiểu (Technical Cleaning)
            df_processed = df_batch.dropDuplicates(['user_id', 'parent_asin', 'timestamp']) \
                .dropna(subset=['user_id', 'parent_asin', 'rating']) \
                .filter((F.col("text").isNotNull()) & (F.trim(F.col("text")) != "")) \
                .withColumn("rating", F.col("rating").cast(DoubleType())) \
                .withColumn("verified_purchase", F.col("verified_purchase").cast(BooleanType())) \
                .withColumn("helpful_vote", F.col("helpful_vote").cast(LongType())) # Giữ nguyên timestamp như ban đầu (long) và cast helpful_vote thành LongType

            # C. Lưu vào Drive (Định dạng Parquet)
            df_processed.write.mode("append").parquet(FINAL_SAVE_PATH)

            print(f"✅ Đã lưu xong Batch { (i+1) // BATCH_SIZE } ({i+1} dòng)")
            buffer = []

    # Xử lý phần dư cuối cùng
    if buffer:
        spark.createDataFrame(buffer).write.mode("append").parquet(FINAL_SAVE_PATH)

    print(f"🏁 Xong! Dữ liệu Bronze đã sẵn sàng tại: {FINAL_SAVE_PATH}")

In [ ]:
def readData(BASE_PATH, DATA_TYPE, CATEGORY):
    df_bronze = spark.read.parquet(os.path.join(BASE_PATH, DATA_TYPE, CATEGORY))
    print(f"📊 Tổng số dòng hiện có trong tầng Bronze: {df_bronze.count()}")
    df_bronze.printSchema()
    df_bronze.limit(5).toPandas()
    df_bronze.show(truncate=False)

In [ ]:
flow(CATEGORY, DATA_TYPE, BASE_PATH)

🚀 Bắt đầu đẩy dữ liệu vào tầng Bronze: All_Beauty
✅ Đã lưu xong Batch 1 (200000 dòng)
✅ Đã lưu xong Batch 2 (400000 dòng)
✅ Đã lưu xong Batch 3 (600000 dòng)
🏁 Xong! Dữ liệu Bronze đã sẵn sàng tại: /content/drive/MyDrive/bigData/bronze/review/All_Beauty


In [ ]:
readData(BASE_PATH, DATA_TYPE, CATEGORY)

📊 Tổng số dòng hiện có trong tầng Bronze: 696007
root
 |-- helpful_vote: long (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)

+------------+-----------+------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+----------------------------+-----------------+
|helpful_vote|parent_asin|rating|text                                                                                                                                                                        